# 🧠 The Universal Pandas Problem-Solving Framework: Complete Substep Breakdown

Here is a detailed, intuitive breakdown of every single substep in the **5-Step Reverse Engineering Framework**, explaining what it means, why it matters, and how you apply it.

---

Always start with thorough scenario comprehension, then work backwards from what the stakeholder wants to see:

<pre style="background: transparent !important; background-color: transparent !important; border: none !important; font-family: 'Courier New', Courier, monospace; font-size: 13px; line-height: 1.25; color: inherit; padding: 0; margin: 15px 0;">
┌────────────────────────────────────────────────────────┐
│ STEP 0: SCENARIO DECODING & RECONNAISSANCE (Groundwork)│
│ • Read the case scenario 10 times thoroughly          │
│ • Break words & sentences into discrete business rules │
│ • Extract & inspect all tables: rows, cols, data types │
│ • (Revisit & re-verify after defining Step 1 contract) │
└──────────────────────────┬─────────────────────────────┘
                           ▼
┌────────────────────────────────────────────────────────┐
│ STEP 1: TARGET OUTPUT CONTRACT (The Destination)       │
│ • What columns and metrics are required?               │
│ • What does exactly 1 row represent (Grain)?           │
│ • What is the final layout? (Wide matrix vs Long table)│
└──────────────────────────┬─────────────────────────────┘
                           ▼
┌────────────────────────────────────────────────────────┐
│ STEP 2: RAW DATA AUDIT (The Starting Point)            │
│ • Which datasets contain the raw fields?               │
│ • Identify quirks: Nulls, string types, join keys      │
└──────────────────────────┬─────────────────────────────┘
                           ▼
┌────────────────────────────────────────────────────────┐
│ STEP 3: 9-STEP MASTER PIPELINE (The Transformation)    │
│ • Ingest/Flatten ➔ Clean/Cast ➔ Filter Early ➔         │
│   Pre-Aggregate ➔ Relational Merge ➔ Window/Bin ➔      │
│   Group/Agg ➔ Derive/Markov ➔ Reshape/Rank/Sort        │
│ • Fast 2-Block Interview Execution Pattern             │
└──────────────────────────┬─────────────────────────────┘
                           ▼
┌────────────────────────────────────────────────────────┐
│ STEP 4: SANITY AUDIT & VERIFICATION (The Guarantee)    │
│ • Assert unique rows, valid metric bounds (0-100%)     │
│ • Guard against 0-division and join duplicates         │
└────────────────────────────────────────────────────────┘
</pre>

---

# 🔎 STEP 0: SCENARIO DECODING & RAW SCHEMA RECONNAISSANCE (The Groundwork)
> **Core Idea:** 90% of interview errors happen because candidates start writing code before deeply understanding the prompt and the shape of raw data. Do not rush into code!

### 🔹 Substep 0.1: Read the Case Scenario 10 Times
* **What it means:** Read through the stakeholder brief at least **10 times** until the end-to-end business narrative (fraud surge, liquidity exposure, AML transaction structuring, credit migration) is crystal clear in your mind.
* **Why it matters:** Fast reading leads to missed edge cases (e.g. "only approved transactions", "distinct customers", "exclude sandbox test accounts").

---

### 🔹 Substep 0.2: Break Down & Decode Words and Paragraph Lines
* **What it means:** Dissect the scenario paragraph sentence-by-sentence and line-by-line:
  - Break complex sentences into discrete business rules.
  - Translate stakeholder jargon into logical conditions (e.g. *"high-velocity bot activity"* $\rightarrow$ `latency_ms <= 60` and `(is_rooted_jailbroken == True | network_type == 'Tor-Proxy')`).
* **Why it matters:** Ensures every business requirement maps to a concrete filter, join, or aggregation condition.

---

### 🔹 Substep 0.3: Extract Tables & Audit Data Shapes and Types
* **What it means:** First identify every table you need to deal with, and inspect:
  - **Rows & Columns:** Check the total row count and column list for each table.
  - **Data Types:** Check timestamp strings, numeric amounts, categorical flags, and booleans (`0/1` vs `'True'/'False'`).
  - **Sample Data:** Inspect sample rows (`df.head(3)`) to see actual values and formats.
* **Why it matters:** Prevents syntax errors, string-type math failures, and silent join mismatch bugs.

---

### 🔹 Substep 0.4: Move to Step 1 & Revisit to Re-Verify
* **What it means:** Proceed to define your **Step 1: Target Output Contract** (columns, grain, layout). Then, **immediately loop back to Step 0** to re-verify that every line and requirement decoded from the prompt is satisfied by your Step 1 contract.
* **Why it matters:** Closes the feedback loop before writing any pipeline code.

---
---

# 🎯 STEP 1: TARGET OUTPUT (The Destination)
> **Core Idea:** You cannot build a bridge if you don't know where the other side of the river is. Before writing any code, define the exact deliverable.

### 🔹 Substep 1.1: Target Schema (Required Columns & Metrics)
* **What it means:** List the exact names and types of columns the business stakeholder wants to see.
* **How to do it:** Read the prompt and list them out:
  - Base Dimensions: `['region', 'card_type']`
  - Aggregated Metrics: `['total_transactions', 'fraud_transactions']`
  - Derived Ratios: `['fraud_rate_pct']`
* **Why it matters:** Prevents you from calculating useless columns or forgetting a critical requested metric.

---

### 🔹 Substep 1.2: Target Grain (Level of Detail)
* **What it means:** Ask yourself: *"What does **exactly ONE row** in the final table represent?"*
* **How to do it:**
  - If 1 row = 1 Customer $\rightarrow$ You will need `groupby('customer_id')` or a customer-level table.
  - If 1 row = 1 Region $\times$ Card Type $\rightarrow$ You will need `groupby(['region', 'card_type'])`.
  - If 1 row = 1 Day $\rightarrow$ You will need `resample('D')` or `pd.Grouper(freq='D')`.
* **Why it matters:** **Grain is the #1 concept in data analysis.** If you get the grain wrong, your entire calculation will be duplicated or miscalculated.

---

### 🔹 Substep 1.3: Target Visual Layout (Shape)
* **What it means:** Determine how the data should be structured visually.
* **The 3 Shapes:**
  1. **Flat / Tabular:** Standard rows and columns (default from `groupby`).
  2. **Wide Matrix (Cross-tab):** One dimension on rows, one on columns (created via `.pivot()` or `pd.crosstab()`). E.g., Markov transition matrices.
  3. **Long / Tidy:** Key-value pairs `[id_vars, metric_name, metric_value]` (created via `pd.melt()`) for feeding into BI charting tools like Tableau/PowerBI.

---
---

# 🔍 STEP 2: RAW DATA AUDIT (The Starting Point)
> **Core Idea:** Inspect your raw ingredients to see what cleaning or merging is required before you can perform math.

### 🔹 Substep 2.1: Source Mapping
* **What it means:** Map each column needed in Step 1 back to its raw source file.
* **How to do it:**
  - `amount`, `is_fraud`, `card_type` $\rightarrow$ found in `raw_transactions.csv`
  - `account_tier`, `kyc_status` $\rightarrow$ found in `customers.csv`
  - `latency_ms`, `network_type` $\rightarrow$ found in `device_telemetry.jsonl`
  - `fico_score` $\rightarrow$ found in `credit_bureau_scores.xml`
* **Decision:** If fields come from 2 different datasets, you know you **must perform a merge/join or pre-aggregation**.

---

### 🔹 Substep 2.2: Data Cleanliness & Type Auditing
* **What it means:** Inspect raw data for hidden bugs that break calculations silently.
* **What to check:**
  1. **Dates stored as strings:** `'2026-01-01'` cannot do time math $\rightarrow$ convert with `pd.to_datetime()`.
  2. **Whitespace & Casing in categories:** `'Visa '` $\neq$ `'Visa'`, `'prime'` vs `'Prime'` $\rightarrow$ `.str.strip().str.title()`.
  3. **Null values in filter keys:** `status.isna()` $\rightarrow$ handle with `.fillna()` before filtering.

---

### 🔹 Substep 2.3: Join Keys & Cardinality Check
* **What it means:** Check if the joining column (e.g. `customer_id`) has duplicate rows in the dimension table.
* **Why it matters:** If `telemetry` or `customers` has duplicate keys, a `left_join` will **silently explode transaction rows**, artificially multiplying all sums and counts! Pre-aggregate child tables first!

---
---

# 🛠️ STEP 3: BACKWARD TRANSFORMATION PIPELINE (The 9-Step Master Architecture)
> **Core Idea:** Transform raw data into the final deliverable using an orderly, robust 9-step execution pipeline covering 100% of simple and advanced fintech operations.

**`1. Ingest & Flatten ➔ 2. Clean & Cast ➔ 3. Filter Early ➔ 4. Pre-Aggregate ➔ 5. Relational Merge ➔ 6. Window & Bin ➔ 7. Group & Aggregate ➔ 8. Derive & Transition ➔ 9. Reshape, Rank & Sort`**

<pre style="background: transparent !important; background-color: transparent !important; border: none !important; font-family: 'Courier New', Courier, monospace; font-size: 13px; line-height: 1.25; color: inherit; padding: 0; margin: 15px 0;">
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ 1. INGEST & FLATTEN      : Multi-format parsing (JSONL/XML/DAT/YAML), .explode()       │
│ 2. CLEAN & CAST          : Strip whitespace, title-case, coerce numerics, parse dates │
│ 3. FILTER EARLY          : Discard out-of-scope records early to maximize speed        │
│ 4. PRE-AGGREGATE         : Collapse 1-to-Many child tables to 1:1 grain before merge   │
│ 5. RELATIONAL MERGE      : Safe 1-to-1 joins / Anti-joins with validate='1:1'          │
│ 6. WINDOW & BINNING      : .rolling(), .ewm(), .shift(), pd.cut(), np.select()        │
│ 7. GROUP & AGGREGATE     : Group by dimensions to reach exact target grain             │
│ 8. DERIVE & TRANSITION   : Calculate KPIs, loss rates, or Markov pd.crosstab() matrices│
│ 9. RESHAPE, RANK & SORT  : .pivot(), .rank(), .sort_values(), .round(2)               │
└────────────────────────────────────────────────────────────────────────────────────────┘
</pre>

---

### 📖 Master Directory of All Methods by Pipeline Step:

| Step # | Pipeline Stage | Primary Focus | Exact Methods & Functions Used |
| :---: | :--- | :--- | :--- |
| **1** | **Ingest & Flatten** | Multi-Format Parsing & Unnesting | `pd.read_csv()`, `pd.read_json(lines=True)`, `pd.json_normalize()`, `ET.parse()`, `pd.read_fwf()`, `yaml.safe_load()`, `.explode()`, `pd.DataFrame.from_dict()` |
| **2** | **Clean & Cast** | Type Coercion & Normalization | `pd.to_datetime()`, `pd.to_numeric(errors='coerce')`, `.astype()`, `.str.strip()`, `.str.title()`, `.str.upper()`, `.str.replace()`, `.fillna()`, `.dropna()`, `.clip()` |
| **3** | **Filter Early** | Fast Row Reduction | `.query()`, `.loc[]`, `.isin()`, `.between()`, `.str.contains()`, `.str.startswith()`, `~` (Bitwise NOT), `&` / `\|` |
| **4** | **Pre-Aggregate** | Grain Alignment (1:1 before Merge) | `.groupby(as_index=False).agg()`, Named Aggregations (`total=('amount', 'sum')`), `.drop_duplicates(subset=[...])`, `.size()` |
| **5** | **Relational Merge** | Safe Joins & Anti-Joins | `.merge(how='inner'\|'left')`, `validate='1:1'`, `validate='m:1'`, `pd.merge_asof()`, Anti-Join Pattern (`indicator=True` + `.query("_merge == 'left_only'")`), `pd.concat()` |
| **6** | **Window & Binning** | Time-Series & Rule Engines | `.rolling(window=N)`, `.ewm(span=N)`, `.shift(N)`, `.pct_change()`, `.cumsum()`, `pd.Grouper(freq='D')`, `pd.cut()`, `pd.qcut()`, `np.select()`, `np.where()`, Island-and-Gap `(cond != cond.shift()).cumsum()` |
| **7** | **Group & Aggregate** | Target Grain Summary | `.groupby(['dim1', 'dim2']).agg()`, `nunique`, `.transform()`, `.filter(lambda g: ...)` |
| **8** | **Derive & Transition**| KPI Ratios & Markov Probabilities | `.assign(lambda d: ...)`, `pd.crosstab(normalize='index')`, Defensive division `(a / b * 100).fillna(0.0)`, `.diff()` |
| **9** | **Reshape, Rank & Sort**| Final Layout & Presentation | `.pivot()`, `.melt()`, `.unstack()`, `.stack()`, `.rank(method='dense')`, `.nlargest()`, `.sort_values()`, `.round(2)`, `.reset_index(drop=True)` |

---

### 🧠 Mental Model: Methods Used at Multiple Stages (The Grain Boundary)

> **Why do methods like `.assign()`, `.query()`, `.sort_values()`, and `.fillna()` appear at both the beginning AND the end of a pipeline?**

Every Pandas pipeline crosses **one critical turning point**: the **`.groupby().agg()`** Grain Boundary.

<pre style="background: transparent !important; background-color: transparent !important; border: none !important; font-family: 'Courier New', Courier, monospace; font-size: 13px; line-height: 1.25; color: inherit; padding: 0; margin: 15px 0;">
                      ┌──────────────────────────────────────────────┐
                      │             WORLD 1: RAW ROWS                │
                      │  (Individual transactions, events, users)   │
                      └──────────────────────┬───────────────────────┘
                                             ▼
                                ⚡ THE GROUPBY CHECKPOINT ⚡
                                             ▼
                      ┌──────────────────────────────────────────────┐
                      │            WORLD 2: SUMMARY ROWS             │
                      │       (Aggregated segments & totals)         │
                      └──────────────────────────────────────────────┘
</pre>

| Method | Role in World 1 (BEFORE GroupBy) | Role in World 2 (AFTER GroupBy) |
| :--- | :--- | :--- |
| **`.assign()`** | **Data Cleaning:** Strip strings, parse dates, cast types.<br>*(e.g., `.assign(tx_date=pd.to_datetime(...))`)* | **KPI Ratios:** Calculate percentages and margins.<br>*(e.g., `.assign(fraud_rate_pct=fraud/total*100))`)* |
| **`.query()`** | **Data Hygiene:** Remove inactive accounts or test records.<br>*(e.g., `.query("status == 'APPROVED'")`)* | **Threshold Filter:** Filter summary groups by KPI.<br>*(e.g., `.query("total_volume > 1_000_000")`)* |
| **`.sort_values()`**| **Time-Series Alignment:** Sort by date *before* rolling/shift.<br>*(e.g., `.sort_values('tx_date')`)* | **Final Presentation:** Sort by highest risk or volume.<br>*(e.g., `.sort_values('total_exposed_usd', ascending=False)`)* |
| **`.fillna()`** | **Dirty Data Defense:** Replace missing raw balances with `0.0`.<br>*(e.g., `.fillna({'account_balance': 0.0})`)* | **Division Defense:** Replace `NaN` from 0-division with `0.0`.<br>*(e.g., `.assign(rate=...).fillna(0.0)`)* |
| **`.round()`** | **Raw Precision:** Round raw currency amounts.<br>*(e.g., `df['amount'].round(2)`)* | **Final Output Polish:** Round all final KPI percentages to 2 decimals.<br>*(e.g., `.round(2)` at the very bottom)* |

### 🔹 Operations 1 & 2: Ingest, Flatten & Type-Cast
Parse multi-format datasets (JSONL, XML, CSV) and cast types:

In [ ]:
import pandas as pd
import numpy as np

# 1. Ingest multi-format data (CSV & JSON Lines)
raw_transactions = pd.read_csv('data/raw_transactions.csv')
customers = pd.read_csv('data/customers.csv')
telemetry = pd.read_json('data/device_telemetry.jsonl', lines=True)

# 2. Clean whitespace and cast types
raw_transactions['tx_date'] = pd.to_datetime(raw_transactions['created_at'])
customers['customer_id'] = customers['customer_id'].astype(str).str.strip()
customers['risk_tier'] = customers['risk_tier'].astype(str).str.strip().str.title()
customers['account_balance'] = pd.to_numeric(customers['account_balance'], errors='coerce').fillna(0.0)
print("✅ Datasets Loaded & Cast:", raw_transactions.shape, customers.shape, telemetry.shape)

### 🔹 Operations 3 & 4: Filter Early & Pre-Aggregate Child Sub-Tables
Filter out-of-scope rows and collapse 1-to-Many child records to 1:1 grain *before* merging:

In [ ]:
# Pre-aggregate child telemetry table to 1 row per customer (Prevents Join Row Explosion)
flagged_telemetry = (
    telemetry
    .assign(
        customer_id=lambda d: d['customer_id'].astype(str).str.strip(),
        latency_ms=lambda d: pd.to_numeric(d['latency_ms'], errors='coerce'),
        is_rooted=lambda d: d['is_rooted_jailbroken'].astype(bool)
    )
    .query("latency_ms <= 60 and (network_type == 'Tor-Proxy' or is_rooted == True)")
    .groupby('customer_id', as_index=False)
    .agg(avg_bot_latency=('latency_ms', 'mean'))
)
flagged_telemetry.head(2)

### 🔹 Operation 5: Safe Relational Merge (1:1 Grain Alignment)
Join cleaned customer dimensions with pre-aggregated attack telemetry:

In [ ]:
joined_exposure = customers.merge(flagged_telemetry, on='customer_id', how='inner', validate='1:1')
joined_exposure.head(2)

### 🔹 Operations 6, 7 & 8: Windowing, Continuous Binning, GroupBy & Derive KPIs
Perform FICO binning (`pd.cut`), time-series rolling (`.rolling`), or target grain aggregations:

In [ ]:
# Group & Aggregate exposure metrics
exposure_summary = (
    joined_exposure
    .groupby(['risk_tier', 'has_crypto_wallet'], as_index=False)
    .agg(
        attacked_customers_count=('customer_id', 'nunique'),
        total_exposed_balance_usd=('account_balance', 'sum'),
        avg_session_latency_ms=('avg_bot_latency', 'mean')
    )
    .round(2)
    .sort_values(by='total_exposed_balance_usd', ascending=False)
    .reset_index(drop=True)
)
exposure_summary

---
### ⚡ The Fast 2-Block Interview Approach (Collapsing the 9 Steps)

> **Core Idea:** In live 20-30 minute coding interviews, you solve Step 3 by organizing all 9 operations into **2 clean, high-speed blocks**:

<pre style="background: transparent !important; background-color: transparent !important; border: none !important; font-family: 'Courier New', Courier, monospace; font-size: 13px; line-height: 1.25; color: inherit; padding: 0; margin: 15px 0;">
┌────────────────────────────────────────────────────────────────────────────────┐
│ 📦 BLOCK 1: Data Preparation & Pre-Aggregation (Operations 1–4)               │
│ Ingest multi-format files, clean strings/types, filter & pre-aggregate child   │
├────────────────────────────────────────────────────────────────────────────────┤
│ ⛓️ BLOCK 2: Master Transformation Chain (Operations 5–9)                       │
│ Safe 1:1 Merge ➔ Bin/Window ➔ Group/Agg ➔ Derive KPIs ➔ Reshape/Rank/Sort      │
└────────────────────────────────────────────────────────────────────────────────┘
</pre>

In [ ]:
# ⚡ Staff-Level 2-Block Live Interview Solution Pattern:

# 📦 BLOCK 1: Ingest, Clean & Pre-Aggregate Child Sub-Table to 1:1 (Operations 1-4)
flagged_telemetry = (
    pd.read_json('data/device_telemetry.jsonl', lines=True)
    .assign(
        customer_id=lambda d: d['customer_id'].astype(str).str.strip(),
        latency_ms=lambda d: pd.to_numeric(d['latency_ms'], errors='coerce'),
        is_rooted=lambda d: d['is_rooted_jailbroken'].astype(bool)
    )
    .query("latency_ms <= 60 and (network_type == 'Tor-Proxy' or is_rooted == True)")
    .groupby('customer_id', as_index=False)
    .agg(avg_bot_latency=('latency_ms', 'mean'))
)

# ⛓️ BLOCK 2: Master Pipeline Chain: Ingest Customers ➔ Merge ➔ Group ➔ Derive ➔ Sort (Operations 5-9)
fast_exposure_summary = (
    pd.read_csv('data/customers.csv')
    .assign(
        customer_id=lambda d: d['customer_id'].astype(str).str.strip(),
        risk_tier=lambda d: d['risk_tier'].astype(str).str.strip().str.title(),
        has_crypto_wallet=lambda d: pd.to_numeric(d['has_crypto_wallet'], errors='coerce').fillna(0).astype(int),
        account_balance=lambda d: pd.to_numeric(d['account_balance'], errors='coerce').fillna(0.0)
    )
    .merge(flagged_telemetry, on='customer_id', how='inner', validate='1:1')
    .groupby(['risk_tier', 'has_crypto_wallet'], as_index=False)
    .agg(
        attacked_customers_count=('customer_id', 'nunique'),
        total_exposed_balance_usd=('account_balance', 'sum'),
        avg_session_latency_ms=('avg_bot_latency', 'mean')
    )
    .round(2)
    .sort_values(by='total_exposed_balance_usd', ascending=False)
    .reset_index(drop=True)
)
fast_exposure_summary

---
---

# 🛡️ STEP 4: SANITY AUDIT & VERIFICATION (The Guarantee)
> **Core Idea:** Don't just assume your code worked—write 3 quick checks to prove that the result is 100% correct.

### 🔹 Substep 4.1: Target Grain Uniqueness Check
Check: Did your groupby produce any duplicate primary keys?

In [ ]:
assert not fast_exposure_summary.duplicated(subset=['risk_tier', 'has_crypto_wallet']).any(), "Error: Duplicate grain found!"
print("✅ Substep 4.1 Passed: No duplicate grain keys!")

### 🔹 Substep 4.2: Range & Boundary Invariants
Check: Are balances and latency numbers positive and valid?

In [ ]:
assert (fast_exposure_summary['total_exposed_balance_usd'] >= 0).all(), "Error: Negative exposed balance!"
assert fast_exposure_summary['avg_session_latency_ms'].between(0, 60).all(), "Error: Latency out of attack threshold bounds!"
print("✅ Substep 4.2 Passed: All balances and latency metrics within valid bounds!")

### 🔹 Substep 4.3: Missing Value Audit
Check: Did any calculations accidentally introduce unexpected NaN values?

In [ ]:
assert fast_exposure_summary.isna().sum().sum() == 0, "Error: Unexpected NaNs found!"
print("✅ Substep 4.3 Passed: Zero unexpected NaNs in final exposure summary!")

---
---

# 💡 Master Summary Table

| Step | Mental Question | Key Action / Operations |
| :--- | :--- | :--- |
| **Step 0 (Groundwork)** | *"What is the business story and what raw schemas exist?"* | Read 10x, decode lines, inspect row/col/types, revisit after Step 1. |
| **Step 1 (Output)** | *"What does the final table look like?"* | Define target schema, 1-row grain, visual layout (flat vs matrix). |
| **Step 2 (Audit)** | *"What raw tables and quirks do I have?"* | Source mapping, type checking, cardinality & join keys. |
| **Step 3 (Pipeline)** | *"How do I transform raw inputs into final output?"* | **9-Step Master Pipeline**: Ingest/Flatten ➔ Clean/Cast ➔ Filter Early ➔ Pre-Aggregate ➔ Merge ➔ Window/Bin ➔ Group/Agg ➔ Derive/Markov ➔ Reshape/Rank/Sort. |
| **Step 4 (Verify)** | *"How do I prove my answer is correct?"* | Assert grain uniqueness, metric ranges, and zero NaNs. |